<a href="https://colab.research.google.com/github/Aditi0912dec/LFBC/blob/main/Agrinet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# libraries


In [ ]:
!pip install pyyaml h5py

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import sklearn
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import KFold, train_test_split
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from sklearn.neural_network import MLPClassifier
from keras import models
from tensorflow.keras.utils import to_categorical
import time
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
import csv
import os
import random
from keras.datasets import fashion_mnist
from sklearn.utils import shuffle
import copy

In [ ]:
import keras
import h5py
from keras.layers import Dropout, Input, Conv2D, Conv3D, MaxPool3D, Flatten, Dense, Reshape, BatchNormalization
from plotly.offline import iplot, init_notebook_mode
from keras.losses import categorical_crossentropy
from keras.optimizers import Adadelta
from keras.models import Sequential, Model
from keras.utils import np_utils
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score
from sklearn.decomposition import IncrementalPCA
from operator import truediv

from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.initializers import Constant
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# writing file

In [ ]:
# file 2  read_write_create_dir

import os
import csv

def read_from_txt(txt_path,beta_value):
  file= open(txt_path,"r")
  file_split=file.read().split()
  b=[]
  client_key_map=[]
  count=0
  for j in range(0,beta_value):
      for i in range(0,len(file_split)):
        count=count+1
        b.append(int(float(file_split[i])))
        #print(b)
        if count%beta_value==0:
          client_key_map.append(b)
          b=[]


  print(client_key_map)
  return client_key_map


#print()


def write_to_txt(client_key_list,txt_path,beta_value):
  with open (txt_path, 'w') as file:
      for i in range(0,len(client_key_list)):
        for j in range(0,beta_value):
          file.write(str(client_key_list[i][j]))
          file.write(" ")
        file.write('\n')
  file.close()
  file= open(txt_path,"r")
  file_split=file.read()
  print(file_split)



# to make the directory and sub directory file

def make_dir(n_devices,parent_dir):
  for client in range(0,n_devices):
      print(f"directory for client: {client}")
      client_dir=f"client_{client}"
      client_dir_path=os.path.join(parent_dir,client_dir)
      if os.path.exists(client_dir_path):
        break
      os.mkdir(client_dir_path)
 ## my version decision tree


# key_ininialize

In [ ]:
# file key_ininialize
def initialize_key(device_list,client_clf_list,beta_value):
  for i in range(0,len(device_list)):
    for j in range(0,beta_value):
      device_list[i].key.append(client_clf_list[i][j])
  return device_list

# model:

In [ ]:
def get_CNN_model_HSI():
  model = Sequential()


  model.add(layers.Conv3D(32,(3,3,3),activation='relu',input_shape=(11,11,20,1),kernel_initializer='he_uniform',bias_initializer=Constant(0.01)))

  model.add(layers.MaxPooling3D((2,2,2)))
  model.add(layers.Conv3D(64,(3,3,3),activation='relu'))
  model.add(layers.MaxPooling3D((2,2,2)))

  model.add(layers.Flatten())

  model.add(layers.Dense(512, activation='relu', kernel_initializer='he_uniform'))
  model.add(layers.Dropout(0.4))

  model.add(layers.Dense(256, activation='relu', kernel_initializer='he_uniform'))
  model.add(layers.Dropout(0.4))

  model.add(layers.Dense(2, activation='softmax'))

	# compile model
  # opt = SGD(learning_rate=0.01, momentum=0.9)
  model.compile(optimizer='SGD',loss='categorical_crossentropy',metrics=['accuracy'])

  return model

# dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from scipy.io import loadmat

HSI = loadmat('/content/drive/MyDrive/MTP_DATA/WHU-Hi-LongKou/WHU_Hi_LongKou.mat')['WHU_Hi_LongKou']
GT = loadmat('/content/drive/MyDrive/MTP_DATA/WHU-Hi-LongKou/WHU_Hi_LongKou_gt.mat')['WHU_Hi_LongKou_gt']


In [ ]:
print(HSI.shape)
print(GT.shape)

(550, 400, 270)
(550, 400)


In [ ]:
# trying formating...


## HSI dataframe:

In [ ]:
print(f'Data Shape: {HSI.shape[:-1]}\nNumber of Bands: {HSI.shape[-1]}')

Data Shape: (550, 400)
Number of Bands: 270


In [ ]:
df = pd.DataFrame(HSI.reshape(HSI.shape[0]*HSI.shape[1], -1))

df.columns = [f'band{i}' for i in range(1, df.shape[-1]+1)]

df['class'] = GT.ravel()

In [ ]:
df.head(5)

,band1,band2,band3,band4,band5,band6,band7,band8,band9,band10,...,band262,band263,band264,band265,band266,band267,band268,band269,band270,class
0,0.119144,0.452328,0.589048,0.711483,0.848857,0.822307,1.020831,1.088326,1.143050,1.161772,...,6.279046,6.698318,7.301254,7.696913,8.261939,8.575315,8.633758,8.413543,8.890121,0
1,0.266055,0.436670,0.684605,0.708308,0.771029,0.803337,0.960599,1.037369,1.260548,1.112069,...,6.227319,6.497162,7.489038,7.888824,8.225585,8.516209,8.744693,8.196497,8.692479,0
2,0.135489,0.552779,0.565573,0.653725,0.708799,0.834761,0.931162,1.007277,1.115141,1.189783,...,6.210227,6.662137,7.323027,7.773304,8.328890,8.723454,8.821000,8.447286,8.567034,0
3,0.175765,0.616654,0.586812,0.719655,0.723561,0.866418,0.906637,0.977693,1.036860,1.068036,...,6.176208,6.670024,7.319337,7.639920,8.610709,8.163558,8.881254,8.785563,8.839517,0
4,0.175710,0.388793,0.595213,0.572375,0.766622,0.809911,0.911719,1.036256,1.195407,1.073012,...,5.913688,6.402612,7.518503,7.206825,7.900758,8.000796,8.446876,8.523228,8.572126,0


In [ ]:
from sklearn.preprocessing import minmax_scale

In [ ]:
# scaling the data
t_df = df[df['class']!=0]
ind = ['band'+str(i) for i in range(1, t_df.shape[-1])]
X = t_df.loc[:, ind]
y = to_categorical(t_df.loc[:, 'class'])
X_scaled = minmax_scale(X, axis = 0);
X_scaled.shape, y.shape

((204542, 270), (204542, 10))

In [ ]:
a_train, a_test, b_train, b_test = train_test_split(X_scaled, y,
                                                    stratify=y,
                                                    test_size=0.30)
print(f"a_train: {a_train.shape}\nb_train: {b_train.shape}\na_test: {a_test.shape}\nb_test: {b_test.shape}")

a_train: (143179, 270)
b_train: (143179, 10)
a_test: (61363, 270)
b_test: (61363, 10)


## dimReduce-iPCA

In [ ]:
# using inc_pca
def inc_PCA_dim_reduction(HSimage,n_Comp):
    reduce_img = np.reshape(HSimage, (-1, HSimage.shape[2]))
    n_batches = 256
    pca_inc = IncrementalPCA(n_components=n_Comp)
    for img_patch in np.array_split(reduce_img, n_batches):
        pca_inc.partial_fit(img_patch)
    X_ipca = pca_inc.transform(reduce_img)
    reduce_img= np.reshape(X_ipca, (HSimage.shape[0],HSimage.shape[1], n_Comp))

    return reduce_img

In [ ]:
HSI = inc_PCA_dim_reduction(HSI,20)


In [ ]:
print(HSI.shape)

(550, 400, 20)


# HSI cube:


In [ ]:
def Pad_Image(HSimage, margin=2):
  # zero padding is performed.
    padded_HSimage = np.zeros((HSimage.shape[0] + 2 * margin, HSimage.shape[1] + 2* margin, HSimage.shape[2]))
    x_pad = margin
    y_pad = margin
    padded_HSimage[x_pad:HSimage.shape[0] + x_pad, y_pad:HSimage.shape[1] + y_pad, :] = HSimage
    return padded_HSimage

In [ ]:
def HSICubes(HSI, GT, WinSize=5, removeZeroLabels = True):
    margin = int((WinSize - 1) / 2)
    zeroPaddedX = Pad_Image(HSI, margin=margin)
    # split patches
    patchesData = np.zeros((HSI.shape[0] * HSI.shape[1], WinSize, WinSize, HSI.shape[2]))
    patchesLabels = np.zeros((HSI.shape[0] * HSI.shape[1]))
    patchIndex = 0
    for r in range(margin, zeroPaddedX.shape[0] - margin):
        for c in range(margin, zeroPaddedX.shape[1] - margin):
            patch = zeroPaddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]
            patchesData[patchIndex, :, :, :] = patch
            patchesLabels[patchIndex] = GT[r-margin, c-margin]
            patchIndex = patchIndex + 1
    if removeZeroLabels:
        patchesData = patchesData[patchesLabels>0,:,:,:]
        patchesLabels = patchesLabels[patchesLabels>0]
        patchesLabels -= 1
    return patchesData, patchesLabels

In [ ]:
def Split_data(HSimage, ground_truth, Test_Ratio, randomState=345):
    x_train, x_test, y_train, y_test = train_test_split(HSimage, ground_truth, test_size=Test_Ratio, random_state=randomState, stratify=ground_truth)
    return x_train, x_test, y_train, y_test

In [ ]:
# X,y,X_test,y_test=Split_data(HSI,GT,0.30)

In [ ]:
HSI,GT=HSICubes(HSI, GT, 11)

In [ ]:
print(HSI.shape)
print(GT.shape)

(204542, 11, 11, 20)
(204542,)


In [ ]:

X,X_test,y,y_test=Split_data(HSI,GT,0.30)

In [ ]:
print(X.shape)
print(y.shape)
print(X_test.shape)
print(y_test.shape)

(143179, 11, 11, 20)
(143179,)
(61363, 11, 11, 20)
(61363,)


# processing data

In [ ]:
# file  get_dataset_labels
def load_dataset():
  print("data loading")
  (X,y),( X_test,y_test) = mnist.load_data()
  print(X.shape,y.shape,X_test.shape,y_test.shape)
  return X,y,X_test,y_test


def data_processing(X):
  # pixel_value in [0,1]
  print("data processing")
  X = X.reshape((X.shape[0],-1))# converting to 1D array
  X = X/255# normalize the input
  print(X.shape)

  return X

def data_processing_HSI(X):
  # pixel_value in [0,1]
  print("data processing")
  X = X.reshape((X.shape[0],-1))# converting to 1D array
  print(X.shape)

  return X


def get_categories(y):
    labels=int(np.unique(y))
    print("number of classes in the dataset: "+str(len(labels)))
    print("classes in the dataset: "+str(labels))

    return labels

In [ ]:
lab=get_categories(y)

number of classes in the dataset: 9
classes in the dataset: [0. 1. 2. 3. 4. 5. 6. 7. 8.]


# data distribution

In [ ]:
# get_distribution
import random
import math
def get_datamap(X,y,labels):
  print("split data as per y label")
  digit_map = {}


  for i in range(len(labels)):
    indices = (y==i)
    Xi = X[indices]
    yi = y[indices]
    digit_map[i] = (Xi,yi)

  for a in range(len(labels)):
    Xa,ya = digit_map[a]
    print (Xa.shape, ya.shape)

  return digit_map


def noniid_distribuition(digit_map,device_list,label,beta_value):
  print("In non iid distribution part 1")
  device_index = 0
  partition_per_class=math.floor((len(device_list)*beta_value)/len(label))
  not_avail=set()
  idx=set(np.arange(0,len(device_list)))
  # getting the keys executing for the first time
  for i in range(0,len(digit_map)):
    print(f"dataset: {i}")
    m=0
    Xi,yi = digit_map[i]
    #print(Xi.shape,yi.shape)
    if(i>=0):
      for d in idx:
        if len(device_list[d].key)==beta_value:
          not_avail.add(d)

    idx=idx.difference(not_avail)
    #print(len(idx))
  #  print(f"selected devices : {sel_device_index}")
    # print(f"left : {len(idx)}")
    # print(f"not avail : {not_avail}")
    # print(f"count not avail : {len(not_avail)}")
    #print(len(not_avail))
    #print(len(idx)+len(not_avail))
  # print(len(idx))
  # sel_device_index=random.choices(list(idx),weights=None,k=partition_per_class)
    if(len(idx)<partition_per_class):
      partition_per_class=len(idx)
    if len(idx)==1:
      break
    sel_device_index=random.sample(list(idx),partition_per_class)
    kf = KFold(n_splits=partition_per_class,shuffle=True)
    for train_indices, test_indices in kf.split(Xi):
    #  print("in csv write ")
  # for s in sel_device_index :
      Xa= Xi[test_indices]
      device_list[sel_device_index[m]].add_data_csv(digit=i,X=Xa)
      device_list[sel_device_index[m]].add_key(digit=i)
      print(f"cient id {sel_device_index[m]} and dataset shape {Xa.shape}")
      m=m+1

################################################
  class_label=list(label)
  print("In non iid distribution part 2")
  if len(idx)==1:
    digit_choice=[x for x in label if x not in set(device_list[i].key)]
    m=random.choice(digit_choice)
    index_0=next(iter(idx))
    device_list[index_0].add_data_csv(digit=m,X=Xi)
    device_list[index_0].add_key(digit=m)
  else:

    for i in idx:
      while len(device_list[i].key)<beta_value:
        digit_choice=[x for x in label if x not in set(device_list[i].key)]
        m=int(random.choice(digit_choice))
        Xi,yi=digit_map[m]
        kf = KFold(n_splits=partition_per_class,shuffle=True)
        for train_indices, test_indices in kf.split(Xi):
          #device_list[i].add_data(digit=m,X=Xi[test_indices])
          device_list[i].add_data_csv(digit=m,X=Xi)
          device_list[i].add_key(digit=m)
          break

######################################################
  #print("device keys")
  for i in range(0, n_devices):
    #print(device_list[i].key)
    print(f"device {i} keys are :{device_list[i].key}")

  return device_list
#######################################################


In [ ]:
hsi_map=get_datamap(X,y,lab)

split data as per y label
(24157, 11, 11, 20) (24157,)
(5862, 11, 11, 20) (5862,)
(2122, 11, 11, 20) (2122,)
(44248, 11, 11, 20) (44248,)
(2906, 11, 11, 20) (2906,)
(8298, 11, 11, 20) (8298,)
(46939, 11, 11, 20) (46939,)
(4987, 11, 11, 20) (4987,)
(3660, 11, 11, 20) (3660,)


# get class : ???

In [ ]:
# file class_module
class MyDigitDevice:

  def __init__(self,id=None):
    self.id = id
    print("device id:" +str(id))
    self.key=[]
    self.digit_data = {}
    self.digit_clf = {}
    self.digit_dataset={}
    self.digit_csv={}
    self.class_weights={}


  def add_key(self,digit=None):
    self.key.append(digit)

  def add_data_csv(self,digit=None,X=None):
   # print("in add data")
   # print("id" +str(self.id))
    #print("digit:" +str(digit))
    sub_dir=f"client_{self.id}"
    sub_sub_dir=f"digit_{digit}.csv"
    path=os.path.join(parent_dir,sub_dir,sub_sub_dir)
   # print(path)
    with open(path,"w") as csvfile:
      np.savetxt(csvfile,X,delimiter=",")
      self.digit_csv[digit]=X
    csvfile.close()

# how to get y?? or y is the digit

  def add_data(self,digit=None):
     sub_dir=f"client_{self.id}"
     sub_sub_dir=f"digit_{digit}.csv"
     path=os.path.join(parent_dir,sub_dir,sub_sub_dir)
     #print(path)
     X= np.loadtxt(path, delimiter=",")
     y=np.repeat(digit,X.shape[0])
     self.digit_data[digit]=(X,y)

    #self.digit_data[digit] = (X,y)

  def get_training_dataset(self,sel_digit=None):
    Xcat_list = []
    ycat_list = []
    X_sub=[]
    y_sub=[]
    # print("in get dataset")
    if sel_digit in self.digit_data.keys():
        client_label=list(self.digit_data.keys())
        for i in client_label:
          Xj, yj = self.digit_data[i]# 10 parts of data with each client
       #   print(Xj.shape)
          Xj=Xj.reshape((Xj.shape[0],28,28)) # reshape to the original format in my case it was 28 x 28
          # when writting in csv the image had to be flatten but when training the CNN model it has to be restore to the original dimension
          #print(i)
          Xcat_list.append(Xj)
          ycat_list.append(yj)

    Xcat = np.concatenate(Xcat_list)
    ycat = np.concatenate(ycat_list)
    sel_pos = (ycat==sel_digit)
    Xpos = Xcat[sel_pos].copy()
    ypos = np.ones(ycat[sel_pos].shape)
    sel_neg = (ycat!=sel_digit)
    Xneg = Xcat[sel_neg].copy()
    yneg = np.zeros(ycat[sel_neg].shape)

    X = np.concatenate((Xpos,Xneg))
    y = np.concatenate((ypos,yneg))
    self.digit_dataset[sel_digit]=(X,y)
    self.class_weights[sel_digit]={1:y.shape[0]/(2*ypos.shape[0]),0: y.shape[0]/((y.shape[0]-ypos.shape[0])*2)} # for balancing the dataset
  def prepare_clf(self,sel_digit=None,central_model=None):
#        if type=="synthetic":
#         model_path=config.local_syn
#       else :
#         model_path=config.local_non_syn

        central_model_weights=central_model.get_weights()
        X,y=self.digit_dataset[sel_digit]
        X,y=shuffle(X,y,random_state=0)
        y= to_categorical(y,2)
        #pos_weight,neg_weight=self.class_weights[0]
        local_model=self.digit_clf[sel_digit]
        local_model.set_weights(central_model_weights)
        epochs=5
#        local_model.compile(optimizer='SGD',loss= 'categorical_crossentropy', metrics=['accuracy'])
        local_model.fit(x=X, y=y, batch_size=15, epochs=epochs, verbose=0,class_weight=self.class_weights[sel_digit])
 #       sub_dir=f"client_{self.id}"
  #      sub_sub_dir=f"digit_{sel_digit}.h5"
   #     h5_path=os.path.join(model_path,sub_dir,sub_sub_dir)
   #     local_model.save(h5_path)
        #self.digit_dataset[sel_digit]=X,y
        self.digit_clf[sel_digit] = local_model


# file create_device_and_init_model

In [ ]:
# file create_device_and_init_model
def make_device(n_devices):
  device_list = []
 # model_list = []
  for i in range(n_devices):
    mydevice = MyDigitDevice(id=i)
  # print(getattr(mydevice,'id'))
  # break
    device_list.append(mydevice)
   # model_list.append(get_CNN_model())
  return device_list

# add data

In [ ]:
# to get the data added to the class
# file 3: add data
def add_client_data(device_list,client_key_map,beta_value):
    print("In add data for the clients")
    for i in range(0,len(device_list)):
      print("data added for clients : " +str(i))
      for j in range(0,beta_value):
      #print(client_key_map[i][j])
        device_list[i].add_data(digit=client_key_map[i][j])
      # if i==2:
      #   break
      #device_list[i].add_data(digit=client_key_map[i][1])
    print("The digit data key for the clients")
    for i in range(0,len(device_list)):

      print(device_list[i].digit_data.keys())
    return device_list

# build one class classifier on each device

In [ ]:
# build one class classifier on each device
# file 4 : prepare classifier
def prepare_client_training_dataset_and_clf(device_list,label):
  print("Training dataset for the clients")
  label_device_map = { new_list: [] for new_list in range(len(label)) }

  #print (id_clf_map)

  for i in range(len(device_list)):
    print("training dataset prepared for client :" +str(i))
    # if i==2:
    #   break
    my_device = device_list[i]
    client_label_list= my_device.digit_data.keys()
    for j in client_label_list:
    # print(j)
     # my_device.get_training_dataset(sel_digit=j,row=rowlist,col=collist,s=stride)
     # X,y=my_device.digit_dataset[j]
      clf=get_CNN_model_HSI()
      my_device.digit_clf[j]=clf
      my_device.get_training_dataset(sel_digit=j)
      #my_device.prepare_clf(sel_digit=j,X=X,y=y)# digit wise classifier for each client
     # print(f"device : {my_device.id}")
      label_device_map[j].append(my_device.id)
  return label_device_map


# binarize

In [ ]:
# file 5  binarize
def prepare_binarize_dataset(yb,selected_digit):
    #yb=np.array(yb)
    y_binarize=yb.copy()
    sel_pos=np.where(y_binarize==selected_digit)[0]
    #print(sel_pos)
    sel_neg=np.where(y_binarize!=selected_digit)[0]
    #print(sel_neg)
    for i in sel_pos:
      y_binarize[i]=1
    for i in sel_neg:
      y_binarize[i]=0

    return y_binarize

# central_model_init

In [ ]:
# file central_model_init
def init_central_model(labels):
  central_model_list=[]
  for i in range(len(labels)):
    fedavg_central_model=get_CNN_model_HSI()
    fedavg_central_model.compile(optimizer='SGD',loss='categorical_crossentropy',metrics=['accuracy'])
    central_model_list.append(fedavg_central_model)
  return central_model_list

# LOCAL_MODEL

In [ ]:
# load_models
def load_local_models(central_model,selected_device,device_list,label):#device list has index of classiofiers
  print("in load model")
  trained_local_models=[]
 # dataset_cardinality_list=[]
  #print(f"load model for label {label}")
  for i in selected_device:
     # print(str(i))
      my_device=device_list[i]
      #clf=selected_device[i]
     # print(my_device.id)
    #  new_model=model_list[i]
    #  f"local_model_for_client_{my_device.id}_digit_{i}.h5"
    #  sub_dir=f"client_{ my_device.id}"
     # sub_sub_dir=f"digit_{label}.h5"
     # h5_path=os.path.join(parent_dir,sub_dir,sub_sub_dir)

      my_device.prepare_clf(label,central_model)# digit wise classifier for each client
     # dataset_cardinality_list.append(dataset_cardinality)
      #new_model= tf.keras.models.load_model(h5_path)
      new_model=my_device.digit_clf[label]
      # y_prob=new_model.predict(X_val,verbose=0)
      # y_validation=prepare_binarize_dataset(y_val,label)
      # y_pred=[]
      # for i in range(0,y_prob.shape[0]):
      #     # y_pred.append(np.argmax(y_prob[i]))
      # accuracy_list.append(f1_score(y_validation,y_pred))
     # f1_list.append(f1_score(y_validation,y_pred,average='macro'))

      #trained_local_models.append(my_device.digit_clf[label])
      trained_local_models.append(new_model)

  return trained_local_models

# AVERAGING

In [ ]:
# averaging
def model_weight_ensemble(members,central_model):
    print("in average")
    n_layers = len(members[0].get_weights())
    # create an set of average model weights
    avg_model_weights = list()
    for layer in range(n_layers):
      #print(layer)
      # collect this layer from each model
      layer_weights = np.array([model.get_weights()[layer] for model in members])
      # weighted average of weights for this layer
      avg_layer_weights = np.average(layer_weights, axis=0)
      # store average layer weights
      avg_model_weights.append(avg_layer_weights)

    #central_model.compile(loss='categorical_crossentropy', optimizer='SGD', metrics=['accuracy'])
    central_model.set_weights(avg_model_weights)# setting the weights of the central model as the average weight of the local model
   # central_model.save(f"central_model_{i}.h5")
    #print(f"ensemble of weights done for label {i}")
    return central_model

# SELECT_client

In [ ]:
def select_client_per_round(label_device_map): # in each round per label 2 clients selected
  selected_device=[]
  for i in range(0,2):
    for i in range(0, len(label_device_map)):
      device=random.sample(label_device_map[i],k=2)
      selected_device.append(device)
  return selected_device

# update_central

In [ ]:
# update_central

def update_central_model(selected_device,central_model,device_list):
 # global m
  print("in update")
  new_central_model_list=[]
  for i in range(0,10):
    #accuracy_list=[]
   # print(f"m value:{m}")
   # print(selected_device[m])
    members=load_local_models(central_model[i],selected_device[i],device_list,i)
   # m=m+1
    # for j in range(0,len(members)):
    #   label_device_map[i][j].digit_clf=members[j]
    new_central_model_list.append(model_weight_ensemble(members,central_model[i]))


  return new_central_model_list

# make_prediction

In [ ]:
# make_prediction
def get_prediction(central_model_list,X_test=None,y_test=None):
  #print("In prediction")
  y_prediction=[]
  avrg_accuracy=0
  for i in range(0,len(central_model_list)):
   # y_test_binarize=prepare_binarize_dataset(y_test,i)
    y_prob_pred=central_model_list[i].predict(X_test,verbose=0)# central model pred
    y_pred=[y_prob_pred[i][1] for i in range(0,len(y_prob_pred))]# to get label specific acc
  #  corr_pred=sum(1 for k,j in zip(y_pred,y_test_binarize) if k==j)# label specific acc
  #  accuracy=round(corr_pred*100/y_test.shape[0],3)
    #print(f"accuracy for central classifier {i} is {accuracy}")
   # avrg_accuracy=avrg_accuracy+accuracy
    y_prediction.append(y_pred)
 # print(f"average accuracy:{avrg_accuracy}")
 # print(len(central_model_list))
  #print(f"mean accuracy of all the clf: {round(avrg_accuracy/len(central_model_list),3)}")
  return y_prediction

# testing

In [ ]:
# test_central
def testing(y_prediction,y_test,round):
  print("in testing")
  label=[0,1,2,3,4,5,6,7,8]
  corr_pred=0
 # print("In testing")
  for i in range(y_test.shape[0]):
    result=[sublist[i] for sublist in y_prediction]
    final_pred=np.argmax(result)
    if final_pred==y_test[i]:
      corr_pred=corr_pred+1
  print(f"accuracy in round {round+1} is :{corr_pred*100/y_test.shape[0]}")
  #return corr_pred*100/y_test.shape[0]


In [ ]:
from statistics import mode

# fedova

In [ ]:
# call_fedova
def fedova(select_client,central_model_list,X_test,y_test,device_list):
  round=range(0,5)
  accuracy_list=[]
  avrg_acc_list=[]

 # with open(result_txt,"a") as f:
   # f.write(title)
   # f.write('\n')
  for i in round:
        print(f"round {i+1}")

      #  f.write("round: "+str(i+1))
       # f.write('\n')
        new_central_model_list= update_central_model(select_client,central_model_list,device_list)
        y_pred=get_prediction(new_central_model_list,X_test,y_test)
     #   avrg_acc_list.append(avrg_accuracy)
        testing(y_pred,y_test,i)
        #accuracy_list.append(testing(y_pred,y_test,i))
     #   one_cycle_acc=one_cycle_fedova(X_test,label_device_map,select_client)
     #   dataframe.loc[len(dataframe.index)]=[accuracy_list[i],avrg_accuracy]
     #   f.write("accuarcy of multiclass classifier:"+str(accuracy_list[i]))
     #   f.write('\n')
     #   f.write("aacuracy bt majority voting: "+str(one_cycle_acc))
     #   f.write('\n')
     #   dataframe.to_csv(dataframe_path,index=False)
        # if accuracy_list[i]>70.00:
        #  break
        central_model_list=new_central_model_list
 # return central_model_list

# main _play_gnd

In [ ]:
n_devices=10
beta_value=2
parent_dir="/content/drive/MyDrive/dataset"
result_txt='/content/drive/MyDrive/demo.txt'
key_path='/content/drive/MyDrive/client_key.txt'
syn_dataframe='/content/drive/MyDrive/syn.csv'
non_syn_dataframe='/content/drive/MyDrive/non_syn.csv'
#json_path='/content/drive/MyDrive/client_key.json'


test_X_dir='/content/drive/MyDrive/test_X.csv'
test_y_dir='/content/drive/MyDrive/test_y.csv'

In [ ]:
print(parent_dir)

/content/drive/MyDrive/dataset


In [ ]:
X,X_test,y,y_test=Split_data(HSI,GT,0.30)

In [ ]:
print(X.shape)
print(y.shape)
print(X_test.shape)
print(y_test.shape)

(143179, 11, 11, 20)
(143179,)
(61363, 11, 11, 20)
(61363,)


In [ ]:
X_test=data_processing(X_test)

data processing
(61363, 2420)


In [ ]:
X=data_processing(X)

data processing
(143179, 2420)


In [ ]:
with open(test_X_dir,"w") as csvfile:
  np.savetxt(csvfile,X_test,delimiter=",")
csvfile.close()

In [ ]:
with open(test_y_dir,"w") as csvfile:
  np.savetxt(csvfile,y_test,delimiter=",")
csvfile.close()

In [ ]:
labels=get_categories(y)

number of classes in the dataset: 9
classes in the dataset: [0. 1. 2. 3. 4. 5. 6. 7. 8.]


In [ ]:
make_dir(n_devices,parent_dir)

directory for client: 0
directory for client: 1
directory for client: 2
directory for client: 3
directory for client: 4
directory for client: 5
directory for client: 6
directory for client: 7
directory for client: 8
directory for client: 9


In [ ]:
digit_map=get_datamap(X,y,labels)

split data as per y label
(24157, 2420) (24157,)
(5862, 2420) (5862,)
(2122, 2420) (2122,)
(44248, 2420) (44248,)
(2906, 2420) (2906,)
(8298, 2420) (8298,)
(46939, 2420) (46939,)
(4987, 2420) (4987,)
(3660, 2420) (3660,)


In [ ]:
device_list=make_device(n_devices)

device id:0
device id:1
device id:2
device id:3
device id:4
device id:5
device id:6
device id:7
device id:8
device id:9


In [ ]:
device_list=noniid_distribuition(digit_map,device_list,labels,beta_value)

In non iid distribution part 1
dataset: 0
cient id 3 and dataset shape (12079, 2420)
cient id 6 and dataset shape (12078, 2420)
dataset: 1
cient id 4 and dataset shape (2931, 2420)
cient id 9 and dataset shape (2931, 2420)
dataset: 2
cient id 2 and dataset shape (1061, 2420)
cient id 5 and dataset shape (1061, 2420)
dataset: 3
cient id 1 and dataset shape (22124, 2420)
cient id 5 and dataset shape (22124, 2420)
dataset: 4
cient id 9 and dataset shape (1453, 2420)
cient id 6 and dataset shape (1453, 2420)
dataset: 5
cient id 2 and dataset shape (4149, 2420)
cient id 0 and dataset shape (4149, 2420)
dataset: 6
cient id 7 and dataset shape (23470, 2420)
cient id 0 and dataset shape (23469, 2420)
dataset: 7
cient id 3 and dataset shape (2494, 2420)
cient id 4 and dataset shape (2493, 2420)
dataset: 8
cient id 1 and dataset shape (1830, 2420)
cient id 8 and dataset shape (1830, 2420)
In non iid distribution part 2
device 0 keys are :[5, 6]
device 1 keys are :[3, 8]
device 2 keys are :[2, 5]

In [ ]:
client_key_list=[]

In [ ]:
for i in range(0,n_devices):
  client_key_list.append(device_list[i].key)
  print(device_list[i].key)


[5, 6]
[3, 8]
[2, 5]
[0, 7]
[1, 7]
[2, 3]
[0, 4]
[6, 3.0]
[8, 1.0]
[1, 4]


In [ ]:
print(client_key_list)

[[5, 6], [3, 8], [2, 5], [0, 7], [1, 7], [2, 3], [0, 4], [6, 3.0], [8, 1.0], [1, 4]]


In [ ]:
write_to_txt(client_key_list,key_path,beta_value)

5 6 
3 8 
2 5 
0 7 
1 7 
2 3 
0 4 
6 3.0 
8 1.0 
1 4 



# something called part 2 ????...

In [ ]:
print(X.shape)

(143179, 2420)


In [ ]:
print(X_test.shape)

(61363, 2420)


In [ ]:
print(X_test[0])

[ 2.79021375e-01 -3.40583708e-02  9.67669943e-03 ... -8.29672935e-04
  1.22870672e-04 -5.43390777e-04]


In [ ]:
X_test_new=X_test.reshape((X_test.shape[0],11,11,20))


In [ ]:
X_test_new.shape

(61363, 11, 11, 20)

In [ ]:
device_list=add_client_data(device_list,client_key_list,beta_value)

In add data for the clients
data added for clients : 0
data added for clients : 1
data added for clients : 2
data added for clients : 3
data added for clients : 4
data added for clients : 5
data added for clients : 6
data added for clients : 7
data added for clients : 8
data added for clients : 9
The digit data key for the clients
dict_keys([5, 6])
dict_keys([3, 8])
dict_keys([2, 5])
dict_keys([0, 7])
dict_keys([1, 7])
dict_keys([2, 3])
dict_keys([0, 4])
dict_keys([6, 3.0])
dict_keys([8, 1.0])
dict_keys([1, 4])


# giving data to CM ??.. not known

In [ ]:
import copy
central_model_list=init_central_model(labels)

# creating n number of central models where n=number of labels

# FEDOVA

In [ ]:
m=0

label_device_map=prepare_client_training_dataset_and_clf(device_list,labels)